<a href="https://colab.research.google.com/github/ambika-1513/Computer-vision-learning/blob/main/Neural_Network_Training_on_GPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

In [2]:
torch.manual_seed(42)

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {device}")

Using Device: cuda


In [21]:
df = pd.read_csv("/content/fashion-mnist_train.csv")

In [22]:
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [23]:
df.shape

(60000, 785)

In [24]:
X = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [27]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state= 22)

In [28]:
X_train = X_train/255.0
X_test = X_test/255.0

In [30]:
class CustomDataset:
  def __init__(self, features, labels):
    self.features = features
    self.labels = labels

  def __len__(self):
    return len(self.features)

  def __getitem__(self, idx):
    return self.features[idx], self.labels[idx]

In [31]:
train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)

In [34]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [35]:
len(train_loader), len(test_loader)

(1500, 375)

In [40]:
class NeuralNetwork(nn.Module):
  def __init__(self, num_features):
    super().__init__()
    self.network = nn.Sequential(
        nn.Linear(num_features, 128),
        nn.ReLU(),
        nn.Linear(128, 64),
        nn.ReLU(),
        nn.Linear(64, 10)
    )
  def forward(self,x):
    return self.network(x)


In [41]:
learning_rate = 0.1
epochs = 100

In [42]:
model = NeuralNetwork(X_train.shape[1])
model = model.to(device)
optimizer = optim.SGD(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

In [44]:
for epoch in range(epochs):
  total_epoch_loss = 0
  for batch_features, batch_labels in train_loader:
    batch_features = batch_features.to(device).float()
    batch_labels = batch_labels.to(device)
    optimizer.zero_grad()
    outputs = model(batch_features)
    loss = criterion(outputs, batch_labels)
    loss.backward()
    optimizer.step()
    total_epoch_loss += loss.item()
  print(f"Epoch: {epoch+1}, Loss: {total_epoch_loss/len(train_loader)}")

Epoch: 1, Loss: 0.6417597307364146
Epoch: 2, Loss: 0.4281853208839893
Epoch: 3, Loss: 0.38425802035629747
Epoch: 4, Loss: 0.3571051267882188
Epoch: 5, Loss: 0.33707536687205236
Epoch: 6, Loss: 0.31961776688694954
Epoch: 7, Loss: 0.30667998164892196
Epoch: 8, Loss: 0.29594812923297287
Epoch: 9, Loss: 0.28603197063754005
Epoch: 10, Loss: 0.2739727932016055
Epoch: 11, Loss: 0.26768976790457966
Epoch: 12, Loss: 0.2582230835730831
Epoch: 13, Loss: 0.2547489096472661
Epoch: 14, Loss: 0.2467468163954715
Epoch: 15, Loss: 0.23963557822505632
Epoch: 16, Loss: 0.2361239752850185
Epoch: 17, Loss: 0.22839323641111453
Epoch: 18, Loss: 0.2229084464510282
Epoch: 19, Loss: 0.22115746448809903
Epoch: 20, Loss: 0.21250146031255523
Epoch: 21, Loss: 0.21029885558908185
Epoch: 22, Loss: 0.20577654097229242
Epoch: 23, Loss: 0.19862930098672707
Epoch: 24, Loss: 0.1970453139077872
Epoch: 25, Loss: 0.19066547514498233
Epoch: 26, Loss: 0.18817073825001718
Epoch: 27, Loss: 0.18365713384374976
Epoch: 28, Loss: 0.1

In [45]:
model.eval()

NeuralNetwork(
  (network): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [48]:
total= 0
correct = 0
with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    batch_features = batch_features.to(device).float()
    batch_labels = batch_labels.to(device)
    outputs = model(batch_features)
    _, predicted = torch.max(outputs.data, 1)
    total += batch_labels.size(0)
    correct += (predicted == batch_labels).sum().item()
print(correct/total)

0.8874166666666666
